In [0]:
# ===============================================
# Notebook: Build fact_sales from Dimension Tables
# ===============================================

from pyspark.sql.functions import col, to_date

# --------------------------------------
# 1. Leitura da staging (arquivo bruto de vendas)
# --------------------------------------
filename_sales = "/FileStore/bronze/abi_bus_case1_beverage_sales_20210726.csv"

df_sales_raw = spark.read \
    .option("header", True) \
    .option("encoding", "utf-8") \
    .option("sep", "\t") \
    .csv(filename_sales)

# Converter todos os nomes de colunas para UPPERCASE logo após a leitura
df_sales_raw = df_sales_raw.toDF(*[c.upper() for c in df_sales_raw.columns])

# --------------------------------------
# 2. Leitura das dimensões
# --------------------------------------
dim_brand = spark.table("beverage_analytics.dim.dim_brand")
dim_region = spark.table("beverage_analytics.dim.dim_region")
dim_channel = spark.table("beverage_analytics.dim.dim_channel")

# --------------------------------------
# 3. Tratamento e junção das dimensões
# --------------------------------------
fact_sales = df_sales_raw \
    .withColumn("volume", col("$ volume").cast("double")) \
    .withColumn("year", col("year").cast("int")) \
    .withColumn("month", col("month").cast("int")) \
    .withColumn("date", to_date(col("date"), "M/d/yyyy")) \
    .join(dim_brand, on=["ce_brand_flvr", "brand_nm"], how="inner") \
    .join(dim_region, on="btlr_org_lvl_c_desc", how="inner") \
    .join(dim_channel, on="trade_chnl_desc", how="inner") \
    .select(
        "date",
        "ce_brand_flvr",
        "brand_nm",
        "btlr_org_lvl_c_desc",
        "trade_chnl_desc",
        "volume",
        "year",
        "month"
    )

# Converter colunas do fato final para lowercase
fact_sales_lower = fact_sales.toDF(*[c.lower() for c in fact_sales.columns])

# --------------------------------------
# 4. Escrita no Unity Catalog (Delta Table)
# --------------------------------------
catalog = "beverage_analytics"
schema = "fact"
table = "fact_sales"


fact_sales_lower.write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("year", "month") \
    .saveAsTable(f"{catalog}.{schema}.{table}")

print(f"Carga concluída com sucesso em: {catalog}.{schema}.{table}")